# RL Jammer DSP acceleration prototype (`RL_Jammer_testing.ipynb`)

This notebook **implements** (not just documents) DSP-chain accelerations:
- vectorized/batched STFT feature extraction,
- cached DSP constants (window/frequency grid),
- decode scheduling for reward computation,
- batched decode calls through `rx_command_iq_broadcast`,
- resample short-circuit when sample rates match.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Sequence, Tuple

import torch

import advanced_link_skdsp_v7_robust as link7
import score_iq_decode as scorer

from accelerated_training_utils import repeat_to_length_mod


In [ ]:
_DSP_CACHE: Dict[Tuple[str, int, int, float], Dict[str, torch.Tensor]] = {}


def _get_stft_constants(device: torch.device,
                        nfft: int,
                        nperseg: int,
                        sample_rate_hz: float) -> Dict[str, torch.Tensor]:
    key = (str(device), int(nfft), int(nperseg), float(sample_rate_hz))
    cached = _DSP_CACHE.get(key)
    if cached is not None:
        return cached

    window = torch.hann_window(nperseg, dtype=torch.float32, device=device)
    fgrid = torch.fft.fftshift(
        torch.fft.fftfreq(nfft, d=1.0 / max(float(sample_rate_hz), 1.0)).to(device)
    ).to(torch.float32)
    out = {"window": window, "fgrid": fgrid}
    _DSP_CACHE[key] = out
    return out


def preprocess_batched_iq_to_stft_feature_fast(
    iq_batch: torch.Tensor,
    sample_rate_hz: float,
    nperseg: int = 128,
    noverlap: int = 96,
    nfft: int = 128,
    out_channels: int = 14,
) -> Dict[str, torch.Tensor]:
    """Vectorized STFT preprocessing for a complex IQ batch [B, T]."""
    if not torch.is_tensor(iq_batch):
        iq_batch = torch.as_tensor(iq_batch)
    if iq_batch.ndim != 2:
        raise ValueError(f"iq_batch must be [B, T], got {tuple(iq_batch.shape)}")

    x = iq_batch.to(dtype=torch.complex64)
    if x.shape[1] < 8:
        x = torch.nn.functional.pad(x, (0, 8 - x.shape[1]))

    x = x - x.mean(dim=1, keepdim=True)
    scale = torch.median(torch.abs(x), dim=1, keepdim=True).values
    x = x / (scale + 1e-6)

    nperseg_eff = int(min(nperseg, max(8, x.shape[1])))
    noverlap_eff = int(min(noverlap, nperseg_eff - 1))
    hop_length = max(1, nperseg_eff - noverlap_eff)

    const = _get_stft_constants(device=x.device, nfft=nfft, nperseg=nperseg_eff, sample_rate_hz=sample_rate_hz)

    z = torch.stft(
        x,
        n_fft=nfft,
        hop_length=hop_length,
        win_length=nperseg_eff,
        window=const["window"],
        center=False,
        return_complex=True,
    )
    z = torch.fft.fftshift(z, dim=1)

    mag = torch.log1p(torch.abs(z)).to(torch.float32)
    phase = torch.angle(z).to(torch.float32)
    real = z.real.to(torch.float32)
    imag = z.imag.to(torch.float32)
    power = (torch.abs(z) ** 2).to(torch.float32)
    power_log = torch.log1p(power)

    f = const["fgrid"][None, :, None]
    denom = power.sum(dim=1, keepdim=True) + 1e-12
    centroid = (f * power).sum(dim=1, keepdim=True) / denom
    centroid_plane = centroid.repeat(1, mag.shape[1], 1) / max(float(sample_rate_hz), 1.0)

    spread = torch.sqrt((((f - centroid) ** 2) * power).sum(dim=1, keepdim=True) / denom + 1e-12)
    spread_plane = (spread / max(float(sample_rate_hz), 1.0)).repeat(1, mag.shape[1], 1)

    flatness = torch.exp(torch.mean(torch.log(torch.clamp(power, min=1e-12)), dim=1, keepdim=True))
    flatness = flatness / (power.mean(dim=1, keepdim=True) + 1e-12)
    flatness_plane = flatness.repeat(1, mag.shape[1], 1)

    frame_power = power.mean(dim=1, keepdim=True)
    frame_power_norm = frame_power / (frame_power.mean(dim=2, keepdim=True) + 1e-12)
    frame_power_plane = frame_power_norm.repeat(1, mag.shape[1], 1)

    delta_t_mag = torch.diff(mag, dim=2, prepend=mag[:, :, :1])
    delta_f_mag = torch.diff(mag, dim=1, prepend=mag[:, :1, :])
    delta_t_power = torch.diff(power_log, dim=2, prepend=power_log[:, :, :1])
    delta_t_phase = torch.diff(phase, dim=2, prepend=phase[:, :, :1])

    sr_plane = torch.full_like(mag, torch.log10(torch.tensor(max(float(sample_rate_hz), 1.0), device=mag.device)) / 10.0)

    feat = torch.stack(
        [
            mag,
            phase,
            centroid_plane,
            sr_plane,
            real,
            imag,
            power_log,
            delta_t_mag,
            delta_f_mag,
            delta_t_phase,
            delta_t_power,
            spread_plane,
            flatness_plane,
            frame_power_plane,
        ][:out_channels],
        dim=1,
    )

    peak_bin = torch.argmax(power.mean(dim=2), dim=1)
    peak_hz = const["fgrid"][peak_bin]
    rx_power = (x.abs() ** 2).mean(dim=1)

    return {
        "feature": feat.to(torch.float32),
        "rx_power": rx_power.to(torch.float32),
        "peak_hz": peak_hz.to(torch.float32),
        "lengths": torch.full((x.shape[0],), float(x.shape[1]), device=x.device, dtype=torch.float32),
    }


In [ ]:
def build_stft_observation_fast(
    *,
    iq1: torch.Tensor,
    iq2: torch.Tensor,
    iq3: torch.Tensor,
    intake_sample_rate_hz: float,
    device: str = "cpu",
) -> Dict[str, List[torch.Tensor]]:
    p1 = preprocess_batched_iq_to_stft_feature_fast(iq1, intake_sample_rate_hz)
    p2 = preprocess_batched_iq_to_stft_feature_fast(iq2, intake_sample_rate_hz)
    p3 = preprocess_batched_iq_to_stft_feature_fast(iq3, intake_sample_rate_hz)
    return {
        "stft_feature_list": [
            p1["feature"].to(device=device, dtype=torch.float32, non_blocking=True),
            p2["feature"].to(device=device, dtype=torch.float32, non_blocking=True),
            p3["feature"].to(device=device, dtype=torch.float32, non_blocking=True),
        ]
    }


In [ ]:
@dataclass
class RewardScheduleConfig:
    decode_every_n_steps: int = 4
    proxy_reward_scale: float = 0.01


def _batched_decode(iq_batch: torch.Tensor, meta_list: Sequence[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if hasattr(link7, "rx_command_iq_broadcast"):
        return link7.rx_command_iq_broadcast(iq_batch, list(meta_list))
    return [link7.rx_command_iq(iq_batch[i], meta_list[i]) for i in range(iq_batch.shape[0])]


def compute_rewards_scheduled_batched(
    *,
    jam_batch: Sequence[Dict[str, Any]],
    samples: Sequence[Dict[str, Any]],
    jammer_sampling_freq: float,
    device: str,
    step_idx: int,
    sched: RewardScheduleConfig,
) -> Tuple[torch.Tensor, int, int]:
    """Batched reward computation with decode scheduling + resample short-circuit."""
    dev = torch.device(device)
    do_decode = (step_idx % max(1, int(sched.decode_every_n_steps)) == 0)

    jammed_rows: List[torch.Tensor] = []
    metas: List[Dict[str, Any]] = []
    proxy_rewards: List[torch.Tensor] = []

    for jam_item, sample in zip(jam_batch, samples):
        tx_iq = jam_item["tx_iq"].to(dtype=torch.complex64)
        whole_iq = sample["whole_iq"].to(dtype=torch.complex64)
        whole_meta = sample["whole_meta"]
        sr_out = float(whole_meta["sample_rate_hz"])

        if abs(float(jammer_sampling_freq) - sr_out) < 1e-6:
            tx_rs = tx_iq
        else:
            tx_rs = link7.resample_iq(tx_iq, fs_in_hz=float(jammer_sampling_freq), fs_out_hz=sr_out)

        tx_rs = repeat_to_length_mod(tx_rs, int(whole_iq.shape[0]))
        tx_rs = torch.as_tensor(tx_rs[: whole_iq.shape[0]], dtype=torch.complex64)

        jammed = whole_iq + tx_rs
        jammed_rows.append(jammed.to(dev))
        metas.append(whole_meta)

        proxy = -sched.proxy_reward_scale * torch.mean(torch.abs(tx_rs) ** 2)
        proxy_rewards.append(proxy.to(torch.float32))

    if not jammed_rows:
        return torch.zeros(0, dtype=torch.float32, device=dev), 0, 0

    if not do_decode:
        return torch.stack(proxy_rewards).to(device=dev), 0, 0

    max_len = max(int(x.numel()) for x in jammed_rows)
    jammed_pad = [torch.nn.functional.pad(x, (0, max_len - int(x.numel()))) for x in jammed_rows]
    jammed_batch = torch.stack(jammed_pad, dim=0).to(device=dev, dtype=torch.complex64)

    results = _batched_decode(jammed_batch, metas)

    scores: List[torch.Tensor] = []
    success = 0
    total = 0
    for r, m in zip(results, metas):
        total += 1
        s = float(scorer.score_decode(r, m)) if r is not None else 0.0
        if s > 0.0:
            success += 1
        scores.append(torch.tensor(s, dtype=torch.float32, device=dev))

    return torch.stack(scores), success, total


In [ ]:
# Example usage sketch inside a loop:
#
# obs = build_stft_observation_fast(
#     iq1=batch["iq1"], iq2=batch["iq2"], iq3=batch["iq3"],
#     intake_sample_rate_hz=jammer_sampling_freq, device=device,
# )
# action_t, value_t, logp_t = policy.get_action_value_logp(obs)
# jam_batch = ...
# rewards_t, success, total = compute_rewards_scheduled_batched(
#     jam_batch=jam_batch,
#     samples=active_samples,
#     jammer_sampling_freq=jammer_sampling_freq,
#     device=device,
#     step_idx=global_step,
#     sched=RewardScheduleConfig(decode_every_n_steps=4, proxy_reward_scale=0.01),
# )
